In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("..")
SILVER_DIR = BASE / "silver_data"
REPORT_DIR = BASE / "reports"
SILVER_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

In [ ]:
df = pd.read_csv("../shipments_realistic.csv")
print("=== PROFILING SHIPPER (raw shipment-level) ===")
print("Shape:", df.shape)
print("Duplicate shipper_id rows:", df["shipper_id"].duplicated().sum())
print("Tổng ô thiếu:", df.isna().sum().sum())

=== PROFILING SHIPPER (raw shipment-level) ===
Shape: (566067, 22)
Duplicate shipper_id rows: 565987
Tổng ô thiếu: 0


## 0. Gộp về 1 dòng / shipper_id

In [ ]:
SHIPPER_COLS = ["shipper_id", "shipper_name", "shipper_phone", "shipper_gender", "shipper_age",
                "shipper_marital_status", "shipper_education", "shipper_company", "shipper_vehicle",
                "shipper_experience_years", "shipper_rating", "delivery_success_rate",
                "average_delivery_time", "working_shift", "join_date", "city", "region", "district"]

consistency_check = df.groupby("shipper_id")[SHIPPER_COLS[1:]].nunique()
inconsistent = consistency_check[(consistency_check > 1).any(axis=1)]
print("Số shipper có thuộc tính KHÔNG hằng định qua các lượt giao hàng:", len(inconsistent))
if len(inconsistent) > 0:
    inconsistent.to_csv(REPORT_DIR / "_review_shipper_inconsistent.csv")
    raise ValueError("Có shipper dữ liệu không hằng định - cần xử lý thủ công trước khi dedupe.")

df = df[SHIPPER_COLS].drop_duplicates(subset="shipper_id").reset_index(drop=True)
print("Sau dedupe:", df.shape)

Số shipper có thuộc tính KHÔNG hằng định qua các lượt giao hàng: 0
Sau dedupe: (80, 18)


## 1. Missing Data

In [ ]:
bad = df[df["shipper_id"].isna()].copy()
print("shipper_id missing:", len(bad))
if len(bad):
    bad.to_csv(REPORT_DIR / "_rejected_shipper_missing_id.csv", index=False)
    df = df[df["shipper_id"].notna()].copy()
print("Missing by column:\n", df.isna().sum())

shipper_id missing: 0
Missing by column:
 shipper_id                  0
shipper_name                0
shipper_phone               0
shipper_gender              0
shipper_age                 0
shipper_marital_status      0
shipper_education           0
shipper_company             0
shipper_vehicle             0
shipper_experience_years    0
shipper_rating              0
delivery_success_rate       0
average_delivery_time       0
working_shift               0
join_date                   0
city                        0
region                      0
district                    0
dtype: int64


## 2. Outlier / Domain rule

In [ ]:
invalid_rating = df[(df["shipper_rating"] < 0) | (df["shipper_rating"] > 5)].copy()
invalid_age = df[(df["shipper_age"] < 18) | (df["shipper_age"] > 70)].copy()
if len(invalid_rating):
    invalid_rating.to_csv(REPORT_DIR / "_rejected_shipper_invalid_rating.csv", index=False)
if len(invalid_age):
    invalid_age.to_csv(REPORT_DIR / "_rejected_shipper_invalid_age.csv", index=False)

df.loc[(df["shipper_rating"] < 0) | (df["shipper_rating"] > 5), "shipper_rating"] = np.nan
df.loc[(df["shipper_age"] < 18) | (df["shipper_age"] > 70), "shipper_age"] = np.nan
print("Invalid rating:", len(invalid_rating))
print("Invalid age:", len(invalid_age))

Invalid rating: 0
Invalid age: 0


## 3. Inconsistency + kiểu dữ liệu

In [ ]:
text_cols = ["shipper_id", "shipper_name", "shipper_gender", "shipper_marital_status",
             "shipper_education", "shipper_company", "shipper_vehicle", "working_shift"]
for col in text_cols:
    df[col] = df[col].astype("string").str.strip()

df["shipper_phone"] = df["shipper_phone"].astype("string").str.strip()
df["join_date"] = pd.to_datetime(df["join_date"], errors="coerce")
df["shipper_age"] = pd.to_numeric(df["shipper_age"], errors="coerce").astype("Int64")
df["shipper_experience_years"] = pd.to_numeric(df["shipper_experience_years"], errors="coerce").astype("Int64")
df["shipper_rating"] = pd.to_numeric(df["shipper_rating"], errors="coerce")
df["delivery_success_rate"] = pd.to_numeric(df["delivery_success_rate"], errors="coerce")
df["average_delivery_time"] = pd.to_numeric(df["average_delivery_time"], errors="coerce").astype("Int64")
print(df.dtypes)

## 3b. Quy đổi city/region/district → zip (FK GEOGRAPHY)

In [ ]:
geo = pd.read_csv("../geography.csv")
geo["zip"] = pd.to_numeric(geo["zip"], errors="coerce").astype("Int64")

geo_sorted = geo.sort_values("zip")
combo_to_zip = geo_sorted.groupby(["city", "region", "district"])["zip"].first().reset_index()
combo_count = geo.groupby(["city", "region", "district"]).size().reset_index(name="n_matching_zip")

df = df.merge(combo_to_zip, on=["city", "region", "district"], how="left")
df = df.merge(combo_count, on=["city", "region", "district"], how="left")

df[["shipper_id", "city", "region", "district", "zip", "n_matching_zip"]].to_csv(
    REPORT_DIR / "shipper_zip_assignment_log.csv", index=False)

print("Số shipper có >1 zip khớp (zip là suy đoán):", (df["n_matching_zip"] > 1).sum(), "/", len(df))
print("Số shipper không tìm được zip khớp:", df["zip"].isna().sum())

df = df.drop(columns=["city", "region", "district", "n_matching_zip"])

Số shipper có >1 zip khớp (zip là suy đoán): 80 / 80
Số shipper không tìm được zip khớp: 0


## 4. Missing sau xử lý domain rule

In [ ]:
for col in ["shipper_rating", "shipper_age"]:
    if df[col].isna().any():
        med = df[col].median()
        df[col] = df[col].fillna(med)
        if col == "shipper_age":
            df[col] = df[col].round().astype("Int64")

print("Missing after processing:\n", df.isna().sum())

Missing after processing:
 shipper_id                  0
shipper_name                0
shipper_phone               0
shipper_gender              0
shipper_age                 0
shipper_marital_status      0
shipper_education           0
shipper_company             0
shipper_vehicle             0
shipper_experience_years    0
shipper_rating              0
delivery_success_rate       0
average_delivery_time       0
working_shift                0
join_date                   0
zip                          0
dtype: int64


## 5. FK và Validation

In [ ]:
geo_zips = set(geo["zip"].dropna())
orphans = df[~df["zip"].isin(geo_zips)].copy()
if len(orphans):
    orphans.to_csv(REPORT_DIR / "_rejected_shipper_orphan_zip.csv", index=False)

assert df["shipper_id"].notna().all(), "SHIPPER: shipper_id còn thiếu"
assert df["shipper_id"].is_unique, "SHIPPER: shipper_id bị trùng"
assert df["zip"].notna().all(), "SHIPPER: zip còn thiếu"
assert (~df["zip"].isin(geo_zips)).sum() == 0, "SHIPPER: có orphan FK zip"
assert df["shipper_rating"].between(0, 5).all(), "SHIPPER: rating ngoài [0,5]"
assert df["shipper_age"].between(18, 70).all(), "SHIPPER: age ngoài [18,70]"
assert df["join_date"].notna().all(), "SHIPPER: join_date không parse được"
print("SHIPPER: validation đạt yêu cầu Silver.")

SHIPPER: validation đạt yêu cầu Silver.


In [ ]:
df.to_csv(SILVER_DIR / "SHIPPER.csv", index=False)
log = [
    ["SHIPPER", "Dedupe", len(df), "Gộp 566,067 dòng shipment về 80 shipper duy nhất, đã kiểm chứng hằng định"],
    ["SHIPPER", "Missing/keys", len(df), "Không bịa khóa; kiểm tra shipper_id"],
    ["SHIPPER", "Domain rules", len(df), "Rating [0,5], age [18,70]"],
    ["SHIPPER", "Inconsistency/type", len(df), "Chuẩn hóa text, date, ID/phone; zip giữ kiểu Int64"],
    ["SHIPPER", "FK zip", len(df), "Quy đổi city/region/district->zip (giả định: chọn zip nhỏ nhất khớp, xem shipper_zip_assignment_log.csv)"],
    ["SHIPPER", "Validation", len(df), "PK, FK, rating, age, date đạt"],
]
pd.DataFrame(log, columns=["table","step","rows_after","result"]).to_csv(REPORT_DIR / "shipper_log.csv", index=False)
print("Đã xuất:", SILVER_DIR / "SHIPPER.csv")

Đã xuất: ../silver_data/SHIPPER.csv
